# Counterparty name = its own bank — `pkg_cpty_name_eq_fi.ipynb`

**Question (Graph stories planning chat):** counterparties such as `124003116-0000000001027136629` are named "ALLY BANK"
and their FI is ALLY BANK. Is that a data quality issue, or the bank's own account at the bank?

**What this checks**
1. The example id: name, FI, categories, direction, number of PNC customers, ticket size, months.
2. How many counterparty ids have a name that matches their own FI (exact / core / truncated prefix), and their dollars.
3. Behaviour of those ids: who pays whom, how many PNC customers, typical ticket. Many customers paying small regular
   amounts *to* the bank = loan / card / mortgage payments, i.e. real.
4. The case name-based mapping gets **wrong**: the name is a bank, but the account's routing number belongs to a
   different bank ("ALLY BANK" with an account at JPMorgan). These are the false MANAGES links.
5. Routing-prefix consistency: does each 9-digit routing prefix map to one FI name in the staging data?

**Inputs** (cdi_tm): `pkg_cpty_month` (name / FI per counterparty per month), `pkg_edge_monthly`, `pkg_cust_dim`.
Optional §6 reads one month of `dsihd01p_dsi.neo4j_payments` for the example id. Nothing is written.

In [ ]:
# §0  Setup
import warnings, html as _html
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
import numpy as np
import pandas as pd
for _alias, _target in [("object0", np.object_), ("bool8", np.bool_), ("int0", np.intp),
                        ("uint0", np.uintp), ("bool", np.bool_)]:
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
from IPython.display import display, HTML
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName("pkg_cpty_name_eq_fi")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "4g").config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g").config("spark.executor.memoryOverhead", "2g")
         .config("spark.dynamicAllocation.enabled", "true").config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true").config("spark.sql.adaptive.skewJoin.enabled", "true")
         .enableHiveSupport().getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", "2000")

DB           = "bdahd01p_dlcdi1_cdi_tm"
T_CPTY_MONTH = f"{DB}.pkg_cpty_month"
T_EDGE_M     = f"{DB}.pkg_edge_monthly"
T_CUST_DIM   = f"{DB}.pkg_cust_dim"
T_STAGING    = "dsihd01p_dsi.neo4j_payments"

EXAMPLE_ID   = "124003116-0000000001027136629"     # from the chat (routing 124003116)
MONTHS       = ["2026-06", "2026-07", "2026-08"]    # window; add months for more history
RUN_STAGING  = False                                # §6: raw rows for the example id (scans one month of staging)
STAGING_MONTH = "2026-08"

# words removed to get a bank's "core" name (ALLY BANK → ALLY, HUNTINGTON NATIONAL BANK → HUNTINGTON)
BANK_WORDS = ["BANK", "BANKS", "NA", "NATIONAL", "ASSOCIATION", "ASSN", "FSB", "SSB", "TRUST", "COMPANY", "CO",
              "CORP", "CORPORATION", "INC", "THE", "OF", "AND", "FEDERAL", "SAVINGS", "CREDIT", "UNION", "FINANCIAL",
              "BANCORP", "BANKING", "USA", "US"]

def _fmt(v, kind):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return ""
    if kind == "pct":
        return "0" if abs(v) < 5e-7 else (f"{v:.3f}" if abs(v) < 1 else f"{v:.2f}")
    return f"{v:,.0f}" if kind == "int" else f"{v:,.1f}" if kind == "f1" else f"{v:,.2f}"

def fmt_df(df):
    out = df.copy()
    for c in out.columns:
        s = out[c]
        if pd.api.types.is_bool_dtype(s) or not pd.api.types.is_numeric_dtype(s):
            out[c] = s.where(s.notna(), "") if not pd.api.types.is_bool_dtype(s) else s
            continue
        kind = ("pct" if "%" in str(c) else "f1" if "$M" in str(c) or "$B" in str(c)
                else "int" if (s.dropna() % 1 == 0).all() or str(c).startswith("$") else "f2")
        out[c] = [_fmt(v, kind) for v in s]
    return out

_CSS = ("<style>.pkgt{border-collapse:collapse;font-size:12px;margin-bottom:4px}"
        ".pkgt th,.pkgt td{padding:2px 9px;border-bottom:1px solid #e3e2de;text-align:right;white-space:nowrap}"
        ".pkgt th{background:#e3e2de;color:#0b0b0b !important;font-weight:600}.pkgn{color:#8a8984;margin:2px 0}</style>")

def show(*blocks):
    parts = []
    for b in blocks:
        if b is None:
            continue
        if isinstance(b, str):
            parts.append(f"<div style='margin:10px 0 2px;font-size:14px'><b>{_html.escape(b)}</b></div>")
        elif b[0] == "__note__":
            parts.append(f"<div class='pkgn'>{_html.escape(b[1])}</div>")
        else:
            parts.append(f"<div style='margin:12px 0 3px'><b>{_html.escape(b[0])}</b></div>")
            parts.append(fmt_df(b[1]).to_html(index=False, escape=True, border=0, classes="pkgt"))
    display(HTML(_CSS + "<div style='font-family:\"IBM Plex Sans\",Arial,sans-serif'>" + "".join(parts) + "</div>"))

def note(t):
    return ("__note__", t)

def norm(c):        # upper, letters and digits only, single spaces
    return F.trim(F.regexp_replace(F.regexp_replace(F.upper(c), r"[^A-Z0-9]+", " "), r"\s+", " "))

def core(c):        # norm without bank words
    n = F.regexp_replace(norm(c), r"\b([A-Z]) (?=[A-Z]\b)", "$1")            # N A → NA
    words = "|".join(sorted(BANK_WORDS, key=len, reverse=True))
    return F.trim(F.regexp_replace(F.regexp_replace(n, rf"\b({words})\b", " "), r"\s+", " "))

def compact(c):     # letters and digits only (for truncated names: HUNTINGTON NATIONAL BA)
    return F.regexp_replace(F.upper(c), r"[^A-Z0-9]", "")
print(f"Spark {spark.version} | window {MONTHS[0]} → {MONTHS[-1]}")

## §1 Per-counterparty summary over the window, with the name-vs-FI match level

In [ ]:
# §1  One row per counterparty id: dominant name and FI (by dollars), categories, match level
cm = spark.table(T_CPTY_MONTH).where(F.col("month").isin(MONTHS))
ids = (cm.groupBy("cpty_id", "cpty_id_src")
         .agg(F.sum(F.col("amount").cast("double")).alias("amt"), F.sum("n_txn").alias("n_txn"),
              F.expr("max_by(struct(cpty_name, fi, cpty_type), amount)").alias("dom"),
              F.countDistinct("cpty_name").alias("n_names"), F.countDistinct("fi").alias("n_fis"),
              F.concat_ws(",", F.sort_array(F.collect_set("category"))).alias("categories"))
         .select("cpty_id", "cpty_id_src", "amt", "n_txn", "n_names", "n_fis", "categories",
                 F.col("dom.cpty_name").alias("name"), F.col("dom.fi").alias("fi"), F.col("dom.cpty_type").alias("cpty_type")))
nc, fc = compact(F.col("name")), compact(F.col("fi"))
ids = (ids.withColumn("routing", F.regexp_extract("cpty_id", r"^(\d{9})-", 1))
          .withColumn("match", F.when(F.col("name").isNull() | F.col("fi").isNull(), "no name or FI")
                               .when(norm(F.col("name")) == norm(F.col("fi")), "exact")
                               .when((F.length(core(F.col("name"))) >= 3) & (core(F.col("name")) == core(F.col("fi"))), "core")
                               .when((F.length(nc) >= 6) & (F.length(fc) >= 6)
                                     & ((F.expr("left(regexp_replace(upper(fi), '[^A-Z0-9]', ''), length(regexp_replace(upper(name), '[^A-Z0-9]', '')))") == nc)
                                        | (F.expr("left(regexp_replace(upper(name), '[^A-Z0-9]', ''), length(regexp_replace(upper(fi), '[^A-Z0-9]', '')))") == fc)),
                                     "prefix (truncated)")
                               .otherwise("different")))
ids = ids.cache()

tot = ids.agg(F.count("*").alias("ids"), F.sum("amt").alias("amt")).first()
lvl = ids.groupBy("match").agg(F.count("*").alias("ids"), F.sum("amt").alias("amt"), F.sum("n_txn").alias("n_txn")).toPandas()
lvl["% ids"] = 100 * lvl.ids / tot["ids"]
lvl["% $"] = 100 * lvl.amt / tot["amt"]
lvl["$M"] = lvl.amt / 1e6
order = {"exact": 0, "core": 1, "prefix (truncated)": 2, "different": 3, "no name or FI": 4}
lvl = lvl.sort_values("match", key=lambda s: s.map(order))
same = ids.where(F.col("match").isin("exact", "core", "prefix (truncated)"))
bycat = (same.groupBy("categories").agg(F.count("*").alias("ids"), F.sum("amt").alias("amt"))
             .orderBy(F.desc("amt")).limit(15).toPandas())
bycat["$M"] = bycat.amt / 1e6
show(f"§1 Counterparty ids in {MONTHS[0]} → {MONTHS[-1]}: {tot['ids']:,} (${tot['amt'] / 1e9:,.1f}B)",
     ("Name vs own FI", lvl[["match", "ids", "% ids", "$M", "% $", "n_txn"]]),
     ("Name = own FI: by category set (top 15 by $)", bycat[["categories", "ids", "$M"]]),
     note("exact = same normalised name; core = same after removing BANK / NA / NATIONAL / TRUST …; "
          "prefix = one is a truncation of the other (≥ 6 characters)."))

## §2 The example id, and its routing number

The routing number is the first 9 digits of the id. If every id under that routing number has the same FI name, the FI
comes from the routing number, not from the counterparty name.

In [ ]:
# §2  Example id: monthly rows, PNC-customer view, and its routing number's other ids
ex_rows = (spark.table(T_CPTY_MONTH).where(F.col("cpty_id") == EXAMPLE_ID)
                .groupBy("month", "category", "cpty_type", "cpty_name", "fi")
                .agg(F.sum("n_txn").alias("txn"), F.sum(F.col("amount").cast("double")).alias("$ amount"))
                .orderBy("month", "category").toPandas())
e = spark.table(T_EDGE_M).where(F.col("month").isin(MONTHS))
ex_edges = (e.where((F.col("dst_id") == EXAMPLE_ID) | (F.col("src_id") == EXAMPLE_ID))
             .withColumn("dir", F.when(F.col("dst_id") == EXAMPLE_ID, "PNC customer pays it").otherwise("it pays PNC customer"))
             .withColumn("cust", F.when(F.col("dst_id") == EXAMPLE_ID, F.col("src_id")).otherwise(F.col("dst_id"))))
ex_dir = (ex_edges.groupBy("month", "dir", "category")
                  .agg(F.countDistinct("cust").alias("PNC customers"), F.sum("n_txn").alias("txn"),
                       F.sum(F.col("amount").cast("double")).alias("$ amount"))
                  .orderBy("month", "dir").toPandas())
ex_dir["$ per txn"] = ex_dir["$ amount"] / ex_dir["txn"]
rt = EXAMPLE_ID.split("-")[0]
same_rt = ids.where(F.col("routing") == rt)
rt_fi = (same_rt.groupBy("fi").agg(F.count("*").alias("ids"), F.sum("amt").alias("amt")).orderBy(F.desc("ids"))
                .limit(10).toPandas())
rt_names = (same_rt.groupBy("name", "match").agg(F.count("*").alias("ids"), F.sum("amt").alias("amt"))
                   .orderBy(F.desc("amt")).limit(15).toPandas())
for d in (rt_fi, rt_names):
    d["$M"] = d.amt / 1e6
show(f"§2 Example {EXAMPLE_ID}",
     ("Rows in pkg_cpty_month (all months)", ex_rows),
     (f"PNC-customer view, {MONTHS[0]} → {MONTHS[-1]}", ex_dir),
     (f"FI names on all ids with routing {rt}", rt_fi[["fi", "ids", "$M"]]),
     (f"Top counterparty names under routing {rt}", rt_names[["name", "match", "ids", "$M"]]))

## §3 Behaviour of name = own-FI counterparties

Customers paying the bank (loan, card, mortgage, brokerage) look like: counterparty is the payee, many PNC customers,
moderate ticket, recurring. A data error would not produce that pattern consistently.

In [ ]:
# §3  Who pays whom, how many PNC customers, ticket size, for the name = own-FI ids
same_ids = F.broadcast(same.select("cpty_id").distinct()) if same.count() < 2_000_000 else same.select("cpty_id")
cd = spark.table(T_CUST_DIM).select(F.col("mdm_id").alias("cust"), F.col("is_tm").cast("boolean").alias("is_tm"))
ed = (e.where((F.col("src_kind") == "CPTY") | (F.col("dst_kind") == "CPTY"))
       .select(F.when(F.col("dst_kind") == "CPTY", F.col("dst_id")).otherwise(F.col("src_id")).alias("cpty_id"),
               F.when(F.col("dst_kind") == "CPTY", F.col("src_id")).otherwise(F.col("dst_id")).alias("cust"),
               (F.col("dst_kind") == "CPTY").alias("cpty_receives"), "month",
               F.col("amount").cast("double").alias("amount"), "n_txn")
       .join(same_ids, "cpty_id", "inner")
       .join(cd, "cust", "left"))
per = (ed.groupBy("cpty_id")
         .agg(F.countDistinct("cust").alias("n_cust"),
              F.countDistinct(F.when(F.col("is_tm"), F.col("cust"))).alias("n_cust_tm"),
              F.sum(F.when(F.col("cpty_receives"), F.col("amount")).otherwise(0.0)).alias("amt_in"),
              F.sum(F.when(~F.col("cpty_receives"), F.col("amount")).otherwise(0.0)).alias("amt_out"),
              F.sum("n_txn").alias("txn"), F.countDistinct("month").alias("months"))
         .join(same.select("cpty_id", "name", "fi", "match", "categories", "amt"), "cpty_id"))
per = per.withColumn("role", F.when((F.col("amt_in") > 0) & (F.col("amt_out") > 0), "both")
                              .when(F.col("amt_in") > 0, "payee (customers pay the bank)")
                              .otherwise("payer (bank pays customers)"))
roles = (per.groupBy("role").agg(F.count("*").alias("ids"), F.sum("amt").alias("amt"),
                                 F.expr("percentile_approx(n_cust, 0.5)").alias("median PNC customers"),
                                 F.expr("percentile_approx(amt / txn, 0.5)").alias("median $ per txn"))
            .toPandas())
roles["$M"] = roles.amt / 1e6
top = (per.groupBy("name", "fi").agg(F.count("*").alias("ids"), F.sum("n_cust").alias("PNC customers (sum)"),
                                    F.sum("n_cust_tm").alias("TM customers (sum)"), F.sum("amt_in").alias("in"),
                                    F.sum("amt_out").alias("out"), F.sum("txn").alias("txn"))
          .withColumn("total", F.col("in") + F.col("out")).orderBy(F.desc("total")).limit(30).toPandas())
top["$M customers pay it"] = top["in"] / 1e6
top["$M it pays customers"] = top["out"] / 1e6
top["$ per txn"] = top.total / top.txn
show("§3 Behaviour of counterparties named like their own bank",
     ("By role", roles[["role", "ids", "$M", "median PNC customers", "median $ per txn"]]),
     ("Top 30 name / FI pairs by dollars", top[["name", "fi", "ids", "PNC customers (sum)", "TM customers (sum)",
                                                "$M customers pay it", "$M it pays customers", "$ per txn"]]))

## §4 Where name-based mapping goes wrong: the name is a bank, the routing number is another bank

The counterparty's name equals the name of some FI in the data, but its own FI (from the routing number) is different —
e.g. "ALLY BANK" with an account at another bank. Name matching would link it to ALLY; key-based mapping links it to the
bank that actually holds the account.

In [ ]:
# §4  Name equals some FI's name, but not its own FI
fi_list = (ids.where(F.col("fi").isNotNull()).select(norm(F.col("fi")).alias("name_n")).distinct()
              .where(F.length("name_n") >= 4).withColumn("name_is_an_fi", F.lit(True)))
other = (ids.where(F.col("match") == "different")
            .withColumn("name_n", norm(F.col("name")))
            .join(F.broadcast(fi_list), "name_n", "inner"))
o_tot = other.agg(F.count("*").alias("ids"), F.sum("amt").alias("amt")).first()
o_top = (other.groupBy("name", "fi").agg(F.count("*").alias("ids"), F.sum("amt").alias("amt"))
              .orderBy(F.desc("amt")).limit(25).toPandas())
o_top["$M"] = o_top.amt / 1e6
o_names = (other.groupBy("name").agg(F.count("*").alias("ids"), F.countDistinct("fi").alias("held at # FIs"),
                                     F.sum("amt").alias("amt"))
                .orderBy(F.desc("ids")).limit(15).toPandas())
o_names["$M"] = o_names.amt / 1e6
show(f"§4 Name is a bank, account at a different bank: {o_tot['ids'] or 0:,} ids, ${(o_tot['amt'] or 0) / 1e6:,.1f}M",
     ("Top 25 by dollars (name → FI from routing number)", o_top[["name", "fi", "ids", "$M"]].rename(columns={"name": "counterparty name", "fi": "FI holding the account"})),
     ("Bank names that appear at the most other banks", o_names[["name", "ids", "held at # FIs", "$M"]]),
     note("Under name-based mapping these would be linked to the bank in their name. Under routing-number mapping they "
          "link to the bank holding the account — the correct MANAGES relationship."))

## §5 Routing-number consistency of the FI field

In [ ]:
# §5  Does each routing number map to one FI name?
rt_all = (ids.where((F.col("routing") != "") & F.col("fi").isNotNull())
             .groupBy("routing").agg(F.countDistinct("fi").alias("n_fi"), F.count("*").alias("ids"),
                                     F.sum("amt").alias("amt"), F.collect_set("fi").alias("fis")))
dist = (rt_all.groupBy(F.least(F.col("n_fi"), F.lit(5)).alias("FI names per routing"))
              .agg(F.count("*").alias("routing numbers"), F.sum("ids").alias("ids"), F.sum("amt").alias("amt"))
              .orderBy("FI names per routing").toPandas())
dist["% ids"] = 100 * dist.ids / dist.ids.sum()
dist["FI names per routing"] = dist["FI names per routing"].map(lambda v: "5+" if v == 5 else str(v))
multi = (rt_all.where(F.col("n_fi") > 1).orderBy(F.desc("ids"))
               .select("routing", "n_fi", "ids", F.concat_ws(" | ", F.slice("fis", 1, 4)).alias("FI names")).limit(15).toPandas())
show("§5 FI names per routing number (ids with a 9-digit routing prefix)",
     ("Distribution", dist[["FI names per routing", "routing numbers", "ids", "% ids"]]),
     ("Routing numbers with more than one FI name", multi))

## §6 Optional: raw staging rows for the example id (one month)

In [ ]:
# §6  Raw rows from neo4j_payments (RUN_STAGING = True; scans one month)
if RUN_STAGING:
    y, m = map(int, STAGING_MONTH.split("-"))
    d0 = f"{STAGING_MONTH}-01"
    d1 = f"{y + (m == 12):04d}-{m % 12 + 1:02d}-01"
    raw = (spark.table(T_STAGING)
                .where((F.col("trans_dt") >= d0) & (F.col("trans_dt") < d1) & (F.col("unq_cpty_acct_id") == EXAMPLE_ID))
                .select("trans_dt", "category", "src_syst", "payment_rail", "cpty_type", "cpty_name", "cpty_fin_entity_name",
                        F.col("mdm_id_pays").isNotNull().alias("customer pays"),
                        F.col("mdm_id_receives").isNotNull().alias("customer receives"),
                        F.col("trans_amt").cast("double").alias("trans_amt"))
                .orderBy("trans_dt").limit(30).toPandas())
    show(f"§6 {T_STAGING}, {STAGING_MONTH}, unq_cpty_acct_id = {EXAMPLE_ID}", ("First 30 rows", raw))
else:
    show(note("§6 skipped (RUN_STAGING = False)"))